# FIFA World Cup 2026 — Notebook 08: Elo Monte Carlo (Postable Forecast)

## About

**Purpose:** Produce the credible, postable champion-probability forecast by running the 10,000-tournament Monte Carlo on the **Elo** ratings instead of the original goal-ratio strengths.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-06-07<br>
**Notes:** Notebook 05's Monte Carlo crowned Morocco (10.2%) because it used the un-adjusted ratio strengths. Notebook 07 showed Elo is materially better (RPS +12.7% on held-out matches) and passes the eye test. This notebook rebuilds Elo on the **full** match history (through 2026 — the train/test split in nb 07 was only for honest scoring), maps Elo → expected goals, and runs the tournament machinery from the shared `match_engine.py` (Dixon–Coles scoreline model + official Round-of-32 bracket). World Cup matches are treated as neutral (no home-advantage term). Output is the forecast you would actually publish.<br>
**Description:** Reads `played_matches.parquet`, `wc_groups.parquet`, `wc_group_fixtures.parquet`.

### Change Control

| Date       | Version | Author      | Changes                                              |
|------------|---------|-------------|------------------------------------------------------|
| 2026-06-07 | 1.0     | Ganapathy K | Initial version                                      |
| 2026-06-09 | 1.1     | Ganapathy K | Shared match_engine: Dixon–Coles + official bracket  |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Imports

In [2]:
import sys
import pandas as pd
import numpy as np
from scipy.stats import poisson
from functools import lru_cache
from pathlib import Path

sys.path.append(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson")
from match_engine import (win_probability as engine_win_probability,
                          build_scoreline_sampler, sample_scorelines,
                          build_round_of_32)

pd.set_option("display.max_rows", 100)

### 1.2 Config

In [3]:
PROCESSED_DATA_DIR = Path(r"D:/Data Science/Visual Studio Code/fifa_wc_2026_poisson/data/processed")
PLAYED_MATCHES_PATH = PROCESSED_DATA_DIR / "played_matches.parquet"
GROUPS_PATH = PROCESSED_DATA_DIR / "wc_groups.parquet"
GROUP_FIXTURES_PATH = PROCESSED_DATA_DIR / "wc_group_fixtures.parquet"
CHAMPION_ODDS_ELO_PATH = PROCESSED_DATA_DIR / "champion_odds_elo.parquet"

INITIAL_RATING = 1500.0
K_FACTOR = 30.0
HOME_ADVANTAGE_ELO = 65.0
MAX_GOALS = 10
RANDOM_SEED = 2026
N_SIMULATIONS = 10000

## 2. Build Elo on the Full History

Same Elo recipe as notebook 07, but walked over **every** played match (no cutoff) so the ratings reflect form right up to 2026. The home bonus applies to non-neutral matches during rating; World Cup predictions later use neutral venues.

In [4]:
played_matches = pd.read_parquet(PLAYED_MATCHES_PATH).sort_values("date").reset_index(drop=True)


def margin_multiplier(goal_difference):
    margin = abs(goal_difference)
    if margin <= 1:
        return 1.0
    if margin == 2:
        return 1.5
    return (11 + margin) / 8


ratings = {}
gap_history, goal_diff_history = [], []
for match in played_matches.itertuples(index=False):
    home_rating = ratings.get(match.home_team, INITIAL_RATING)
    away_rating = ratings.get(match.away_team, INITIAL_RATING)
    home_bonus = 0.0 if match.neutral else HOME_ADVANTAGE_ELO
    rating_gap = home_rating + home_bonus - away_rating
    expected_home = 1.0 / (1.0 + 10.0 ** (-rating_gap / 400.0))

    goal_diff = match.home_score - match.away_score
    actual_home = 1.0 if goal_diff > 0 else (0.5 if goal_diff == 0 else 0.0)
    gap_history.append(rating_gap); goal_diff_history.append(goal_diff)

    change = K_FACTOR * margin_multiplier(goal_diff) * (actual_home - expected_home)
    ratings[match.home_team] = home_rating + change
    ratings[match.away_team] = away_rating - change

slope, intercept = np.polyfit(gap_history, goal_diff_history, 1)
total_goals = (played_matches["home_score"] + played_matches["away_score"]).mean()
print(f"Rated {len(ratings)} teams through {played_matches['date'].max().date()}")
print(f"Supremacy fit: goal_diff = {slope:.5f} * gap + {intercept:.3f} | avg total goals {total_goals:.3f}")

Rated 336 teams through 2026-06-03
Supremacy fit: goal_diff = 0.00580 * gap + 0.233 | avg total goals 2.940


## 3. Load the World Cup Field & Validate

The 48 qualified teams and their groups (notebook 04). Every team must have an Elo rating, or the simulation will fall back to a default and silently mislead.

In [5]:
groups_table = pd.read_parquet(GROUPS_PATH)
group_fixtures = pd.read_parquet(GROUP_FIXTURES_PATH)
team_to_group = dict(zip(groups_table["team"], groups_table["group"]))

missing = [t for t in team_to_group if t not in ratings]
assert not missing, f"WC teams with no Elo rating: {missing}"
print(f"All {len(team_to_group)} World Cup teams have Elo ratings.")
print("\nWC field by Elo:")
wc_elo = pd.Series({t: ratings[t] for t in team_to_group}).sort_values(ascending=False)
for team, rating in wc_elo.head(10).items():
    print(f"  {team:<15} {rating:.0f}")

All 48 World Cup teams have Elo ratings.

WC field by Elo:
  Spain           2169
  Argentina       2161
  France          2115
  Brazil          2065
  Colombia        2040
  Portugal        2030
  England         2029
  Germany         1999
  Netherlands     1991
  Japan           1990


## 4. Elo → Expected Goals → Match Probabilities

World Cup matches are neutral, so no home bonus: the rating gap is just `Elo(A) − Elo(B)`. The supremacy fit turns that gap into a goal difference, split around the average total to give each side's expected goals, then the Poisson grid gives the scoreline distribution. `win_probability` (cached) is the knockout resolver.

In [6]:
def elo_expected_goals(team_a, team_b):
    gap = ratings[team_a] - ratings[team_b]
    supremacy = slope * gap + intercept
    expected_a = max((total_goals + supremacy) / 2.0, 0.05)
    expected_b = max((total_goals - supremacy) / 2.0, 0.05)
    return expected_a, expected_b


@lru_cache(maxsize=None)
def win_probability(team_a, team_b):
    expected_a, expected_b = elo_expected_goals(team_a, team_b)
    return engine_win_probability(expected_a, expected_b)


def knockout_winner(team_a, team_b, rng):
    return team_a if rng.random() < win_probability(team_a, team_b) else team_b

## 5. Group Stage (vectorised) & Bracket

Same fast group-stage qualifier logic as nb 05, but expected goals now come from Elo. The Round-of-32 template and single-elimination progression are unchanged.

In [7]:
team_list = list(team_to_group.keys())
group_of_index = [team_to_group[t] for t in team_list]
home_idx = np.array([team_list.index(t) for t in group_fixtures["home_team"]])
away_idx = np.array([team_list.index(t) for t in group_fixtures["away_team"]])
expected_home = np.array([elo_expected_goals(h, a)[0]
                          for h, a in zip(group_fixtures["home_team"], group_fixtures["away_team"])])
expected_away = np.array([elo_expected_goals(h, a)[1]
                          for h, a in zip(group_fixtures["home_team"], group_fixtures["away_team"])])
group_to_indices = {g: [i for i, gi in enumerate(group_of_index) if gi == g]
                    for g in sorted(set(group_of_index))}
group_stage_sampler = build_scoreline_sampler(expected_home, expected_away)


def simulate_qualifiers(rng):
    home_goals, away_goals = sample_scorelines(group_stage_sampler, rng)
    points = np.zeros(len(team_list)); gf = np.zeros(len(team_list)); ga = np.zeros(len(team_list))
    np.add.at(gf, home_idx, home_goals); np.add.at(gf, away_idx, away_goals)
    np.add.at(ga, home_idx, away_goals); np.add.at(ga, away_idx, home_goals)
    hw = home_goals > away_goals; aw = away_goals > home_goals; dr = home_goals == away_goals
    np.add.at(points, home_idx[hw], 3); np.add.at(points, away_idx[aw], 3)
    np.add.at(points, home_idx[dr], 1); np.add.at(points, away_idx[dr], 1)
    gd = gf - ga

    def rank_key(i):
        return (points[i], gd[i], gf[i])

    winners, runners, third_candidates = {}, {}, []
    for group, indices in group_to_indices.items():
        ordered = sorted(indices, key=rank_key, reverse=True)
        winners[group] = ordered[0]; runners[group] = ordered[1]
        third_candidates.append((group, ordered[2]))
    best_thirds = sorted(third_candidates, key=lambda gi: rank_key(gi[1]), reverse=True)[:8]
    return ({g: team_list[i] for g, i in winners.items()},
            {g: team_list[i] for g, i in runners.items()},
            {g: team_list[i] for g, i in best_thirds})

## 6. Run the Monte Carlo

10,000 full tournaments on the Elo engine. For each team, count how often it reached each milestone.

In [8]:
ROUND_NAMES = ["round_of_16", "quarter_final", "semi_final", "final", "champion"]


def simulate_tournament(rng):
    winners, runners, best_thirds = simulate_qualifiers(rng)
    current = build_round_of_32(winners, runners, best_thirds)
    reached = {}; round_index = 0
    while len(current) >= 1:
        round_winners = [knockout_winner(a, b, rng) for a, b in current]
        for team in round_winners:
            reached[team] = round_index
        if len(round_winners) == 1:
            break
        current = [(round_winners[i], round_winners[i + 1]) for i in range(0, len(round_winners), 2)]
        round_index += 1
    return reached


rng = np.random.default_rng(RANDOM_SEED)
milestones = {name: {team: 0 for team in team_list} for name in ROUND_NAMES}
for _ in range(N_SIMULATIONS):
    reached = simulate_tournament(rng)
    for team, deepest in reached.items():
        for r in range(deepest + 1):
            milestones[ROUND_NAMES[r]][team] += 1

champion_odds = pd.DataFrame({n: pd.Series(c) for n, c in milestones.items()})
champion_odds = (champion_odds / N_SIMULATIONS * 100).round(1)
champion_odds = champion_odds.sort_values("champion", ascending=False)
champion_odds.columns = [c.replace("_", " ").title() + " %" for c in champion_odds.columns]
print(f"Ran {N_SIMULATIONS} tournaments on the Elo engine")
champion_odds.head(20)

Ran 10000 tournaments on the Elo engine


,Round Of 16 %,Quarter Final %,Semi Final %,Final %,Champion %
Spain,85.1,62.2,55.5,37.0,25.8
Argentina,81.7,74.1,61.7,41.6,23.7
France,86.9,58.7,46.5,28.3,17.6
Brazil,70.4,50.6,33.7,18.1,7.7
England,70.4,41.7,20.1,9.3,3.5
Germany,74.9,33.7,20.8,8.6,3.3
Colombia,68.3,37.2,17.6,7.9,3.2
Portugal,73.8,41.3,17.7,8.0,3.1
Ecuador,65.0,27.8,15.5,6.2,2.1
Netherlands,48.1,28.9,12.4,4.9,1.7


## 7. Save the Postable Forecast

In [9]:
champion_odds.to_parquet(CHAMPION_ODDS_ELO_PATH)
print("Top 10 title favourites (Elo Monte Carlo):")
for team, row in champion_odds.head(10).iterrows():
    print(f"  {team:<15} {row['Champion %']:>5}%")

Top 10 title favourites (Elo Monte Carlo):
  Spain            25.8%
  Argentina        23.7%
  France           17.6%
  Brazil            7.7%
  England           3.5%
  Germany           3.3%
  Colombia          3.2%
  Portugal          3.1%
  Ecuador           2.1%
  Netherlands       1.7%
